In [1]:
# ============================================================
# Cell 2: 导入所有必要的库
# ============================================================
import os
import sys
import random
from pathlib import Path
from collections import Counter, defaultdict
import xml.etree.ElementTree as ET

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.patches as patches
from matplotlib.patches import Rectangle
from matplotlib.colors import LinearSegmentedColormap
from PIL import Image
import cv2

import torch
import torch.nn as nn
from ultralytics import YOLO

# 设置中文字体支持（如无中文字体则回退）
plt.rcParams['font.sans-serif'] = ['SimHei', 'Microsoft YaHei', 'DejaVu Sans']
plt.rcParams['axes.unicode_minus'] = False
plt.rcParams['figure.dpi'] = 100

# 固定随机种子
random.seed(42)
np.random.seed(42)
torch.manual_seed(42)

print("所有库导入成功！")
print(f"PyTorch 版本: {torch.__version__}")
print(f"CUDA 可用: {torch.cuda.is_available()}")

所有库导入成功！
PyTorch 版本: 2.5.1+cu121
CUDA 可用: True


---
## 4. 特征图/注意力权重可视化

使用 PyTorch Hook 机制提取 YOLOv8 模型中间层的特征图，观察模型学到了哪些视觉特征。

In [2]:
# ============================================================
# Cell 18: 特征图可视化 - 注册 Hook 提取中间层输出
# ============================================================
# 存储特征图的字典
feature_maps = {}

def get_feature_map_hook(name):
    """创建 forward hook 来捕获特征图"""
    def hook(module, input, output):
        if isinstance(output, (list, tuple)):
            feature_maps[name] = output[0].detach()
        else:
            feature_maps[name] = output.detach()
    return hook

# 注册 hooks - 选择 backbone 的几个关键层
# YOLOv8n 模型结构: model.model 是基础模型
try:
    base_model = model.model.model if hasattr(model.model, 'model') else model.model
    
    hooks = []
    # 尝试在 backbone 的不同 stage 注册 hook
    target_indices = [0, 2, 4, 6, 9]  # 浅层到深层
    
    for idx in target_indices:
        if idx < len(base_model):
            layer = base_model[idx]
            hook_name = f'layer_{idx}'
            try:
                h = layer.register_forward_hook(get_feature_map_hook(hook_name))
                hooks.append(h)
                print(f"已注册 hook: {hook_name} -> {type(layer).__name__}")
            except Exception as e:
                print(f"注册 hook {hook_name} 失败: {e}")
    
    print(f"\n共注册 {len(hooks)} 个 feature hooks")
except Exception as e:
    print(f"注册 hooks 时出错: {e}")
    print("将使用备选方案：直接使用 model.predict 获取特征")

注册 hooks 时出错: name 'model' is not defined
将使用备选方案：直接使用 model.predict 获取特征


In [3]:
# ============================================================
# Cell 19: 运行前向传播并可视化特征图
# ============================================================
# 选择一张测试图像
test_img_path = str(list(val_img_dir.glob('*.jpg'))[0])
print(f"测试图像: {test_img_path}")

# 显示原图
fig_disp, ax_disp = plt.subplots(1, 1, figsize=(8, 6))
img_test = Image.open(test_img_path)
ax_disp.imshow(img_test)
ax_disp.set_title(f'Test Image: {Path(test_img_path).name}', fontsize=12)
ax_disp.axis('off')
plt.show()

# 运行推理（这会触发 hooks）
print("\n正在运行前向传播以提取特征图...")
feature_maps.clear()
with torch.no_grad():
    _ = model(test_img_path, verbose=False)

print(f"提取到 {len(feature_maps)} 层特征图")
for name, fm in feature_maps.items():
    print(f"  {name}: shape = {fm.shape}")

NameError: name 'val_img_dir' is not defined